# Wix RAG Project: Tracing, Evaluating, and Profiling

In this notebook, we will set up a Wix Help Center RAG (Retrieval-Augmented Generation) agent using NVIDIA NeMo Agent Toolkit (NAT). The workflow uses Milvus as the vector store and built-in NAT tools for retrieval, memory, web search, and code generation. We will also cover <a href="https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/observe/index.html">observability</a>, <a href="https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/evaluate.html">evaluation</a>, and <a href="https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/profiler.html">profiling</a> of the workflow.

# Table of Contents

- [0.0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Installing NeMo Agent Toolkit](#installing-nat)
- [1.0) Creating the Wix RAG Workflow](#creating-workflow)
  - [1.0.1) Setting Up Milvus and Ingesting Data](#setup-milvus-ingest)
  - [1.1) Workflow Configuration File](#workflow-config)
  - [1.2) Testing/Verifying Workflow Installation](#verify-tools)
- [2.0) Observing a Workflow with Phoenix](#observe-workflow)
  - [2.1) Updating the Workflow Configuration For Telemetry](#update-config)
  - [2.2) Start Phoenix Server](#start-phoenix)
  - [2.3) Rerun the Workflow](#rerun-workflow)
  - [2.4) Viewing the Trace](#view-trace)
- [3.0) Evaluating a Workflow](#eval-workflow)
  - [3.1) Create an Evaluation Dataset](#eval-dataset)
  - [3.2) Updating the Workflow Configuration](#update-config-again)
  - [3.3) Running the Evaluation](#run-eval)
  - [3.4) Understanding Evaluation Results](#understand-eval)
- [4.0) Profiling a Workflow](#profile-workflow)
  - [4.1) Updating the Workflow Configuration](#update-profiling-workflow)
  - [4.2) Understanding the Profiler Configuration](#understand-profiler-config)
  - [4.3) Running the Profiler](#run-profiler)
  - [4.4) Understanding Profiler Output Files](#understand-profiler-output-files)
- [5.0) Notebook Summary](#summary)
- [6.0) Next Steps](#next-steps)


<a id="setup"></a>
# 0.0) Setup


<a id="prereqs"></a>
## 0.1) Prerequisites

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Python Packages:** `pip`

<a id="api-keys"></a>
## 0.2) API Keys

For this notebook, you will need the following API keys to run all examples end-to-end:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys

Then you can run the cell below:

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("/localhome/local-jilei/NeMo-Agent-Toolkit/.env")

for key in ["NVIDIA_API_KEY", "MEM0_API_KEY", "TAVILY_API_KEY"]:
    assert os.environ.get(key), f"{key} not found in .env"
    print(f"{key} = {os.environ[key][:12]}...")

<a id="installing-nat"></a>
## 0.3) Installing NeMo Agent Toolkit

The recommended way to install NAT is through `pip` or `uv pip`.

First, we will install `uv` which offers parallel downloads and faster dependency resolution.

In [2]:
!pip install uv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


NeMo Agent toolkit can be installed through the PyPI `nvidia-nat` package.

There are several optional subpackages available for NAT. For this example, we will rely on three subpackages:
* The `langchain` subpackage contains useful components for integrating and running within [LangChain](https://python.langchain.com/docs/introduction/).
* The `llama-index` subpackage contains useful components for integrating and running within [LlamaIndex](https://developers.llamaindex.ai/python/framework/).
* The `phoenix` subpackage contains components for integrating with [Phoenix](https://phoenix.arize.com/).
* The `profiling` subpackage contains components common for profiling with NeMo Agent toolkit.

In [3]:
%%bash
cd /localhome/local-jilei/NeMo-Agent-Toolkit
uv pip install -e ".[langchain,llama-index,phoenix,profiling]"

Using Python 3.12.11 environment at: /localhome/local-jilei/.venv


Resolved 229 packages in 5.26s
   Building nvidia-nat @ file:///localhome/local-jilei/NeMo-Agent-Toolkit
      Built nvidia-nat @ file:///localhome/local-jilei/NeMo-Agent-Toolkit
Prepared 1 package in 4.72s
Uninstalled 1 package in 1ms
Installed 1 package in 2ms
 ~ nvidia-nat==1.4.0.dev80+geeb19bc5 (from file:///localhome/local-jilei/NeMo-Agent-Toolkit)


<a id="creating-workflow"></a>
# 1.0) Creating the Wix RAG Workflow

This is a Wix Help Center RAG agent that uses built-in NAT tools to:
- Retrieve information from a Milvus vector store containing Wix Help Center articles
- Rerank retrieved results for better relevance
- Store and recall user preferences via memory
- Search the web for supplementary information
- Generate code when requested

Since all tools used are built-in NAT components (`nat_retriever`, `milvus_retriever`, `milvus_rerank_retriever`, `add_memory`, `get_memory`, `tavily_internet_search`, `code_generation`), we do not need to write custom tool code. We only need to configure them in the workflow YAML file.

In [4]:
!nat workflow create wix_rag

Installing workflow 'wix_rag'...
Workflow 'wix_rag' installed successfully.
Workflow 'wix_rag' created successfully in '/localhome/local-jilei/NeMo-Agent-Toolkit/examples/rag_project/wix_rag'.


A summary of the high-level components are outlined below.

* `configs` (symbolic link to `src/wix_rag/configs`)
* `data` (symbolic link to `src/wix_rag/data`)
* `pyproject.toml` Python project configuration file
* `src`
  * `wix_rag`
    * `__init__.py` Module init file (empty)
    * `configs` Configuration directory for workflow specifications
      * `config.yml` Workflow configuration file
    * `data` Data directory for any dependent files
    * `wix_rag.py` User-defined code for workflow execution
    * `register.py` Automatic registration of project components

<a id="setup-milvus-ingest"></a>
## 1.1) Setting Up Milvus and Ingesting Data

Before configuring the workflow, we need to start the Milvus vector database and ingest the Wix Help Center corpus into it.

**Start Milvus** (skip this step if you already have Milvus running):

In [ ]:
%%bash
cd /localhome/local-jilei/NeMo-Agent-Toolkit
docker compose -f examples/deploy/docker-compose.milvus.yml up -d

**Ingest Wix Help Center data** into the Milvus collection:

In [ ]:
%%bash
cd /localhome/local-jilei/NeMo-Agent-Toolkit
python scripts/ingest_wix_corpus.py --collection_name wix_collection

<a id="workflow-config"></a>
## 1.2) Workflow Configuration File

Below is the workflow configuration file for the Wix RAG agent. It defines:
- **Retrievers**: Milvus vector store retriever with optional reranking
- **Functions**: Built-in NAT tools for retrieval, memory, web search, and code generation
- **LLMs**: NIM-hosted models for the agent, evaluation, and embeddings
- **Workflow**: A ReAct agent that orchestrates the tools
- **Eval**: Evaluation and profiling configuration
- **Optimizer**: Prompt and parameter optimization settings

**Prerequisites**: Ensure Milvus is running at `localhost:19530` with the `wix_collection` already populated.

In [5]:
%%writefile wix_rag/configs/config.yml
# SPDX-FileCopyrightText: Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

        
memory:
  saas_memory:
    _type: mem0_memory

retrievers:
  wix_retriever:
    _type: milvus_retriever
    uri: http://localhost:19530
    collection_name: "wix_collection"
    embedding_model: milvus_embedder
    top_k: 10
    
  wix_retriever_with_rerank:
    _type: milvus_rerank_retriever 
    uri: http://localhost:19530
    collection_name: "wix_collection"
    embedding_model: milvus_embedder
    reranker_model_name: "nvidia/llama-3.2-nv-rerankqa-1b-v2"    
    top_k: 6                        # 最终返回给 Agent 的文档数
    expansion_factor: 3        # 初始检索 5*3=15 个文档进行 rerank

functions:
  wix_retriever_tool:
    _type: nat_retriever
    retriever: wix_retriever
    topic: Retrieve information about Wix features and support
    description: |
      Retrieves information from the Wix Help Center knowledge base.
      Use this tool for questions about website building, features, accounts, domains, or support, even if "Wix" is not explicitly mentioned in the query.
      Assume generic questions (e.g., "how to add a page", "change domain settings") are referring to the Wix platform.
      Topics covered include:
      - Website building and editing
      - Features and functionality
      - Account and settings
      - Apps and integrations
      - Domains and email
      - E-commerce and bookings

  wix_retriever_tool_with_rerank:
    _type: nat_retriever
    retriever: wix_retriever_with_rerank
    topic: Retrieve information about Wix features and support with reranking
    description: |
      Retrieves information from the Wix Help Center knowledge base.
      Use this tool for questions about website building, features, accounts, domains, or support, even if "Wix" is not explicitly mentioned in the query.
      Assume generic questions (e.g., "how to add a page", "change domain settings") are referring to the Wix platform.
      Topics covered include:
      - Website building and editing
      - Features and functionality
      - Account and settings
      - Apps and integrations
      - Domains and email
      - E-commerce and bookings
    
  add_memory:
    _type: add_memory
    memory: saas_memory
    description: |
      Add any facts about user preferences to long term memory. Always use this if users mention a preference.
      The input to this tool should be a string that describes the user's preference, not the question or answer.
  get_memory:
    _type: get_memory
    memory: saas_memory
    description: |
      Always call this tool before calling any other tools, even if the user does not mention to use it.
      The question should be about user preferences which will help you format your response.
      For example: "How does the user like responses formatted?"

  # To use these tools you will need to install the "nvidia-nat[langchain]" package
  web_search_tool:
    _type: tavily_internet_search
    max_results: 5
    # Tavily internet search requires an API Key. You can specify it here, or export the TAVILY_API_KEY environment variable
    # api_key: "{your key goes here}"
  code_generation_tool:
    _type: code_generation
    llm_name: nim_llm
    description: |
      Always call this tool to generate python code. Returns a code snippet which MUST be included in your response.

  prompt_optimizer:
    _type: prompt_init
    optimizer_llm: nim_llm
    system_objective: "Answer questions about Wix using the provided tools and knowledge base."

  prompt_recombiner:
    _type: prompt_recombiner
    optimizer_llm: nim_llm
    system_objective: "Answer questions about Wix using the provided tools and knowledge base."

llms:
  nim_llm:
    _type: nim
    model_name: qwen/qwen3-next-80b-a3b-instruct
    temperature: 0.1
    max_tokens: 4096
    top_p: 0.9
    optimizable_params:
      - temperature
      - top_p
      # - max_tokens
    search_space:
      temperature:
        low: 0.0
        high: 0.3
        step: 0.1
      top_p:
        low: 0.8
        high: 1.0
        step: 0.1
      # max_tokens:
      #   low: 1024
      #   high: 4096
      #   step: 128

  nim_rag_eval_llm:
    _type: nim
    model_name: meta/llama-3.3-70b-instruct
    max_tokens: 1024
    temperature: 0.0
  nim_trajectory_eval_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0
    max_tokens: 1024

embedders:
  milvus_embedder:
    _type: nim
    model_name: nvidia/nv-embedqa-e5-v5
    truncate: "END"

workflow:
  _type: react_agent
  tool_names:
    # - wix_retriever_tool  
    - wix_retriever_tool_with_rerank
    - add_memory
    - get_memory
    - web_search_tool
    - code_generation_tool
  verbose: true
  llm_name: nim_llm
  
  system_prompt: |
    Answer the following questions as best you can, utilizing the provided tools and knowledge base to achieve the objective of answering questions.
    You may ask the human to use the following tools: 

    {tools}.

    To respond, follow this structured approach:
    1. **Initial Assessment**: Evaluate the input question to determine if it can be answered directly or if tool usage is required.

    2. **Tool Usage (if necessary)**: Use the following format exactly to ask the human to use a tool:
      - Question: the input question you must answer 
      - Thought: consider what tool to use from {tools} and how to formulate the action input, thinking about what to do
      - Action: the action to take, should be one of [{tool_names}]
      - Action Input: the input to the action (if there is no required input, include \"Action Input: None\") 
      - Observation: wait for the human to respond with the result from the tool, do not assume the response

    3. **Iterative Tool Usage (if necessary)**: Repeat the Thought, Action, Action Input, and Observation steps as necessary until you have sufficient information to answer the question.

    4. **Final Answer**: Once all necessary information is obtained, use the following format to provide the final answer: 
      - Thought: I now know the final answer
      - Final Answer: the final answer to the original input question
        **Examples**:
        - If the question is about a specific topic, you might use a tool to look up relevant information.
        - If the question is about how to use a specific feature, you might use a tool to find a tutorial or guide.

    **Error Handling**: If tools fail or inputs are insufficient, specify the issue in your Thought process and attempt to find an alternative solution or request more information.

    **Output Schema**: Ensure all final answers are provided in a clear, direct format, following the specified Final Answer structure.

    **Safety and Compliance**: Refuse requests that are unsafe or non-compliant with rules and guidelines.

    You may respond in one of two formats. The above sequence can be used to ask for tools and wait for the response. 
    If you do not need to use a tool, or after asking the human to use any tools and waiting for the human to respond, you might know the final answer, and then provide the final answer as specified. 

    By following this structured approach, you will effectively utilize the provided tools and knowledge base to answer questions while maintaining clarity, precision, and adherence to the specified output schema.","
    Instructions for the ReAct agent. IMPORTANT: You MUST preserve the exact placeholders {tools} and {tool_names} in the prompt, as they are required for the agent to function.

  optimizable_params:
    - system_prompt

  search_space:
    system_prompt:
      is_prompt: true
      prompt_purpose: "Instructions for the ReAct agent. IMPORTANT: You MUST preserve the exact placeholders {tools} and {tool_names} in the prompt, as they are required for the agent to function."
      prompt: |
        Answer the following questions as best you can. You may ask the human to use the following tools:

        {tools}

        You may respond in one of two formats.
        Use the following format exactly to ask the human to use a tool:

        Question: the input question you must answer
        Thought: you should always think about what to do
        Action: the action to take, should be one of [{tool_names}]
        Action Input: the input to the action (if there is no required input, include "Action Input: None")
        Observation: wait for the human to respond with the result from the tool, do not assume the response

        ... (this Thought/Action/Action Input/Observation can repeat N times. If you do not need to use a tool, or after asking the human to use any tools and waiting for the human to respond, you might know the final answer.)
        Use the following format once you have the final answer:

        Thought: I now know the final answer
        Final Answer: the final answer to the original input question


Overwriting wix_rag/configs/config.yml


<a id="verify-tools"></a>
## 1.2) Testing/Verifying Workflow Installation

You can verify the workflow was successfully set up by running the following example:


In [6]:
!nat run --config_file wix_rag/configs/config.yml \
  --input "How do I add a page in Wix?" \
  --input "How do I change my domain settings?"

2026-02-26 07:46:49 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'wix_rag/configs/config.yml'
2026-02-26 07:46:49 - ERROR    - nat.data_models.config:107 - Requested memory type `mem0_memory` not found. Have you ensured the necessary package has been installed with `uv pip install`?
Available memory names:
 - 

2026-02-26 07:46:49 - ERROR    - nat.utils.exception_handlers.schemas:76 - Invalid configuration: memory: Input tag 'mem0_memory' found using discriminator() does not match any of the expected tags: '_ignore/0', '_ignore/1', '0', '1'
Traceback (most recent call last):
  File "/localhome/local-jilei/NeMo-Agent-Toolkit/src/nat/utils/exception_handlers/schemas.py", line 72, in inner_function
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/localhome/local-jilei/NeMo-Agent-Toolkit/src/nat/utils/data_models/schema_validator.py", line 30, in validate_schema
    raise e
  File "/localhome/local-jilei/NeMo-Agent-Toolkit/src/nat/utils

<a id="observe-workflow"></a>
# 2.0) Observing a Workflow with Phoenix

> **Note:** _This portion of the example will only work when the notebook is run locally. It may not work through Google Colab and other online notebook environments._

Phoenix is an open-source observability platform designed for monitoring, debugging, and improving LLM applications and AI agents. It provides a web-based interface for visualizing and analyzing traces from LLM applications, agent workflows, and ML pipelines. Phoenix automatically captures key metrics such as latency, token usage, and costs, and displays the inputs and outputs at each step, making it invaluable for debugging complex agent behaviors and identifying performance bottlenecks in AI workflows.

<a id="update-config"></a>
## 2.1) Updating the Workflow Configuration For Telemetry

We will need to update the workflow configuration file to support telemetry tracing with Phoenix.

To do this, we will first copy the original configuration:

In [ ]:
!cp rag_project/configs/config.yml rag_project/configs/phoenix_config.yml

Then we will append necessary configuration components to the `phoenix_config.yml` file:

In [ ]:
%%writefile -a rag_project/configs/phoenix_config.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: WARN
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: rag_project


<a id="start-phoenix"></a>
## 2.2) Start Phoenix Server

First, we will install Phoenix:

In [ ]:
!uv pip install arize-phoenix

Then, we will ensure the service is publicly accessible:

In [ ]:
%env PHOENIX_HOST=0.0.0.0

Finally, we will start the server:

In [ ]:
%%bash --bg
# phoenix will run on port 6006
phoenix serve

<a id="rerun-workflow"></a>
## 2.3) Rerun the Workflow

Instead of the original workflow configuration, we will run with the updated `phoenix_config.yml` file:

In [ ]:
!nat run --config_file rag_project/configs/phoenix_config.yml \
  --input "How do I add a page in Wix?" \
  --input "How do I change my domain settings?"

<a id="view-trace"></a>
## 2.3) Viewing the trace

You can access the Phoenix server at http://localhost:6006

<a id="eval-workflow"></a>
# 3.0) Evaluating a Workflow

After setting up observability, the next step is to evaluate your workflow's performance against a test dataset. NAT provides a powerful evaluation framework that can assess your agent's responses using various metrics and evaluators.

For detailed information on evaluation, please refer to the [Evaluating NVIDIA NeMo Agent Toolkit Workflows](https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/evaluate.html).


<a id="eval-dataset"></a>
## 3.1) Create an Evaluation Dataset

For evaluating this workflow, we will create a sample dataset based on the Wix Help Center knowledge base.

The dataset contains five test cases covering different Wix-related query types. Each entry contains a question and the expected answer that the agent should provide.


In [ ]:
%%writefile rag_project/data/eval_data.json
[
    {
        "id": "1",
        "question": "Can I start accepting payments on my site while my Wix Payments account is still under verification?",
        "answer": "You can start accepting payments on your site using Wix Payments almost immediately. However, we need to verify your identity before your account can be fully activated."
    },
    {
        "id": "2",
        "question": "I want to know if the Wix store function work for selling services instead of just physical goods",
        "answer": "Wix Stores gives you all the tools you need to create a sleek, professional online store and grow your eCommerce business. Wix Bookings, however, is an easy-to-use scheduling system that lets your members book and pay for services online, so you can focus on managing your business."
    },
    {
        "id": "3",
        "question": "How do I sync the hotel app with my calendars ical link to allow visitors to book available dates on my site?",
        "answer": "To sync your hotel app with your calendars using an iCal link, you can import reservations from external services like Airbnb, HomeAway, and VRBO to your Wix Hotels reservation calendar. This is a one-way sync from iCal to the Wix Calendar."
    },
    {
        "id": "4",
        "question": "I am trying to verify my domain with Google Workspace. I need to add a TXT record to my domains DNS settings. I have confirmed that my domain is connected to Wix via pointing. What to do now?",
        "answer": "To verify your domain with Google Workspace by adding a TXT record, you need to manage your DNS settings with your domain host, not Wix, since your domain is connected via pointing. Log in to your domain host account where your DNS records are managed, add the TXT record provided by Google Workspace to your domain DNS settings, then return to the Google Admin Console and click Verify."
    },
    {
        "id": "5",
        "question": "I want to know how much it would cost to upgrade my email plan.",
        "answer": "To upgrade your email marketing plan: Go to Email Marketing in your site dashboard. Under Monthly Balance click Upgrade. Choose a plan that works best for you and checkout."
    }
]

In [ ]:
!nat info components -t evaluator

<a id="update-config-again"></a>
## 3.2) Updating the Workflow Configuration

Workflow configuration files can contain extra settings relevant for evaluation and profiling.

To do this, we will first copy the original configuration:

In [ ]:
!cp rag_project/configs/config.yml rag_project/configs/config_eval.yml

*Then* we will append necessary configuration components to the `config_eval.yml` file:

In [ ]:
%%writefile -a rag_project/configs/config_eval.yml

eval:
  general:
    output_dir: ./eval_output
    verbose: true
    max_concurrency: 2
    dataset:
        _type: json
        file_path: ./rag_project/data/eval_data.json

  evaluators:
    accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_rag_eval_llm
    groundedness:
      _type: ragas
      metric: ResponseGroundedness
      llm_name: nim_rag_eval_llm
    relevance:
      _type: ragas
      metric: ContextRelevance
      llm_name: nim_rag_eval_llm
    trajectory_accuracy:
      _type: trajectory
      llm_name: nim_trajectory_eval_llm


<a id="run-eval"></a>
## 3.3) Running the Evaluation

The `nat eval` command executes the workflow against all entries in the dataset and evaluates the results using configured evaluators. Run the cell below to evaluate the Wix RAG workflow.


In [ ]:
!nat eval --config_file rag_project/configs/config_eval.yml

<a id="understand-eval"></a>
## 3.4) Understanding Evaluation Results

The `nat eval` command runs the workflow on all entries in the dataset and produces several output files:

- **`workflow_output.json`**: Contains the raw outputs from the workflow for each input in the dataset
- **Evaluator-specific files**: Each configured evaluator generates its own output file with scores and reasoning

#### Evaluation Scores

Each evaluator provides:
- An **average score** across all dataset entries (0-1 scale, where 1 is perfect)
- **Individual scores** for each entry with detailed reasoning
- **Performance metrics** to help identify areas for improvement

All evaluation results are stored in the `output_dir` specified in the configuration file.


<a id="profile-workflow"></a>
## 4.0)  Profiling a Workflow

Profiling provides deep insights into your workflow's performance characteristics, helping you identify bottlenecks, optimize resource usage, and improve overall efficiency.

For detailed information on profiling, please refer to the [Profiling and Performance Monitoring of NVIDIA NeMo Agent Toolkit Workflows](https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/profiler.html).


<a id="update-profiling-workflow"></a>
## 4.1) Updating the Workflow Configuration

Workflow configuration files can contain extra settings relevant for evaluation and profiling.

To do this, we will first copy the original configuration:

In [ ]:
!cp rag_project/configs/config.yml rag_project/configs/config_profile.yml

*Then* we will append necessary configuration components to the `config_profile.yml` file:

In [ ]:
%%writefile -a rag_project/configs/config_profile.yml

eval:
  general:
    output_dir: ./profile_output
    verbose: true
    max_concurrency: 2
    dataset:
        _type: json
        file_path: ./rag_project/data/eval_data.json

    profiler:
        token_uniqueness_forecast: true
        workflow_runtime_forecast: true
        compute_llm_metrics: true
        csv_exclude_io_text: true
        prompt_caching_prefixes:
          enable: true
          min_frequency: 0.1
        bottleneck_analysis:
          enable_nested_stack: true
        concurrency_spike_analysis:
          enable: true
          spike_threshold: 7

  evaluators:
    accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_rag_eval_llm
    groundedness:
      _type: ragas
      metric: ResponseGroundedness
      llm_name: nim_rag_eval_llm
    relevance:
      _type: ragas
      metric: ContextRelevance
      llm_name: nim_rag_eval_llm
    trajectory_accuracy:
      _type: trajectory
      llm_name: nim_trajectory_eval_llm


<a id="understand-profiler-config"></a>
## 4.2) Understanding the Profiler Configuration

We will reuse the same configuration as evaluation.

The profiler is configured through the `profiler` section of your workflow configuration file. It runs alongside the `nat eval` command and offers several analysis options:

#### Key Configuration Options:

- **`token_uniqueness_forecast`**: Computes the inter-query token uniqueness forecast, predicting the expected number of unique tokens in the next query based on tokens used in previous queries

- **`workflow_runtime_forecast`**: Calculates the expected workflow runtime based on historical query performance

- **`compute_llm_metrics`**: Computes inference optimization metrics including latency, throughput, and other performance indicators

- **`csv_exclude_io_text`**: Prevents large text from being dumped into output CSV files, preserving CSV structure and readability

- **`prompt_caching_prefixes`**: Identifies common prompt prefixes that can be pre-populated in KV caches for improved performance

- **`bottleneck_analysis`**: Analyzes workflow performance measures such as bottlenecks, latency, and concurrency spikes
  - `simple_stack`: Provides a high-level analysis
  - `nested_stack`: Offers detailed analysis of nested bottlenecks (e.g., tool calls inside other tool calls)

- **`concurrency_spike_analysis`**: Identifies concurrency spikes in your workflow. The `spike_threshold` parameter (e.g., 7) determines when to flag spikes based on the number of concurrent running functions

#### Output Directory

The `output_dir` parameter specifies where all profiler outputs will be stored for later analysis.


<a id="run-profiler"></a>
## 4.3) Running the Profiler

The profiler runs as part of the `nat eval` command. When properly configured, it will collect performance data across all evaluation runs and generate comprehensive profiling reports.


In [ ]:
!nat eval --config_file rag_project/configs/config_profile.yml

<a id="understand-profiler-output-files"></a>
## 4.4) Understanding Profiler Output Files

Based on the profiler configuration, the following files will be generated in the `output_dir`:

**Core Output Files:**

1. **`all_requests_profiler_traces.json`**: Raw usage statistics collected by the profiler, including:
   - Raw traces of LLM interactions
   - Tool input and output data
   - Runtime measurements
   - Execution metadata

2. **`inference_optimization.json`**: Workflow-specific performance metrics with confidence intervals:
   - 90%, 95%, and 99% confidence intervals for latency
   - Throughput statistics
   - Workflow runtime predictions

3. **`standardized_data_all.csv`**: Standardized usage data in CSV format containing:
   - Prompt tokens and completion tokens
   - LLM input/output
   - Framework information
   - Additional metadata


**Advanced Analysis Files**

4. **Analysis Reports**: JSON files and text reports for any advanced techniques enabled:
   - Concurrency analysis results
   - Bottleneck analysis reports
   - PrefixSpan pattern mining results

These files provide comprehensive insights into your workflow's performance and can be used for optimization and debugging.

**Gantt Chart**

We can also view a Gantt chart of the profile run:

In [ ]:
from IPython.display import Image

Image("profile_output/gantt_chart.png")

**Observability with Phoenix**
- Configured tracing in the workflow configuration
- Started the Phoenix server for real-time monitoring
- Executed workflows with automatic trace capture
- Visualized agent execution flow and LLM interactions


**Evaluation with `nat eval`**
- Created a comprehensive evaluation dataset
- Ran automated evaluations across multiple test cases
- Reviewed evaluation metrics and scores
- Analyzed workflow performance against expected outputs



**Profiling for Performance Optimization**
- Configured advanced profiling options
- Collected performance metrics and usage statistics
- Generated detailed profiling reports
- Identified bottlenecks and optimization opportunities



These three pillars—observability, evaluation, and profiling—work together to provide a complete picture of your agent's behavior, accuracy, and performance, enabling you to build production-ready AI applications with confidence.